# Init


In [6]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 4.4 MB/s eta 0:00:00a 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [55]:
import os
import numpy as np
import pandas as pd
import re
import spacy

from scipy.spatial.distance import euclidean
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DOCS_FOLDER_PATH = os.path.join(os.path.abspath('.'), 'docs')

In [11]:
class TextParser:
    def __init__(self, category_mapping_path: str, folder_path: str = DOCS_FOLDER_PATH, files_format: str = 'txt'):
        self.category_mapping_path = category_mapping_path
        self.folder_path = folder_path
        self.files_format = files_format

    def parce(self) -> pd.DataFrame:
        docs = self.get_files_data()
        categories = pd.read_csv(self.category_mapping_path)
        return docs.merge(categories, on="file_name")

    def get_files_data(self) -> pd.DataFrame:
        docs = {}
        for file_name in os.listdir(self.folder_path):
            if not file_name.endswith(self.files_format):
                continue
            with open(os.path.join(self.folder_path, file_name)) as f:
                docs[file_name] = f.read()
        return pd.DataFrame(docs.items(), columns=['file_name', 'raw_text'])

In [13]:
class TextPreprocessor:
    def __init__(self, lang_model: str = "en_core_web_sm"):
        self.nlp = spacy.load(lang_model)

    def preprocess_text(self, text: str) -> list[str]:
        text = self._remove_non_letters(text)
        return self._lemmatize(text)

    def preprocess_documents(self, docs: dict[str, str]) -> dict[str, list[str]]:
        return {name: self.preprocess_text(text) for name, text in docs.items()}

    @staticmethod
    def _remove_non_letters(text: str) -> str:
        text = text.lower()
        return re.sub(r'[^a-z\s]', '', text)

    def _lemmatize(self, text: str) -> list[str]:
        return [token.lemma_ for token in self.nlp(text) if token.is_alpha and not token.is_stop]

In [14]:
class FrequencyAnalysis:
    def __init__(self, tokens_df: pd.DataFrame):
        self.tokens_df = tokens_df
        self.vectorizer = CountVectorizer()
        self.word_doc_matrix: pd.DataFrame | None = None

    def build_word_document_matrix(self) -> pd.DataFrame:
        docs = self.tokens_df["tokens"].apply(lambda x: " ".join(x))
        matrix = self.vectorizer.fit_transform(docs)
        self.word_doc_matrix = pd.DataFrame(
            matrix.toarray(),
            index=self.tokens_df["file_name"],
            columns=self.vectorizer.get_feature_names_out()
        )
        return self.word_doc_matrix

    def build_word_word_matrix(self) -> pd.DataFrame:
        if self.word_doc_matrix is None:
            self.build_word_document_matrix()
        word_word = np.dot(self.word_doc_matrix.T, self.word_doc_matrix)
        return pd.DataFrame(
            word_word,
            index=self.vectorizer.get_feature_names_out(),
            columns=self.vectorizer.get_feature_names_out()
        )

    def build_document_category_matrix(self) -> pd.DataFrame:
        categories = pd.get_dummies(self.tokens_df['category'])
        categories.index = self.tokens_df['file_name']
        return categories

    @staticmethod
    def euclidean_distance(matrix: pd.DataFrame) -> pd.DataFrame:
        dist = pd.DataFrame(index=matrix.index, columns=matrix.index)
        for i in matrix.index:
            for j in matrix.index:
                dist.loc[i, j] = euclidean(matrix.loc[i], matrix.loc[j])
        return dist

    @staticmethod
    def cosine_similarity_matrix(matrix: pd.DataFrame) -> pd.DataFrame:
        sim = cosine_similarity(matrix)
        return pd.DataFrame(sim, index=matrix.index, columns=matrix.index)


In [57]:
class NaiveBayesClassifier:
    def __init__(self):
        self.pipeline = Pipeline([
            ('vectorizer', CountVectorizer()),
            ('classifier', MultinomialNB())
        ])

    def train(self, texts: pd.Series, categories: pd.Series) -> None:
        self.pipeline.fit(texts, categories)

    def predict(self, texts: list[str]) -> list[str]:
        return self.pipeline.predict(texts).tolist()

    def predict_proba(self, texts: list[str]) -> list[list[float]]:
        return self.pipeline.predict_proba(texts).tolist()

class NumericNaiveBayesClassifier:
    def __init__(self):
        self.pipeline = make_pipeline(
            StandardScaler(),
            GaussianNB()
        )

    def train(self, X: np.ndarray, y: pd.Series) -> None:
        self.pipeline.fit(X, y)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.pipeline.predict(X)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return self.pipeline.predict_proba(X)

In [16]:
class SVDReducer:
    def __init__(self, n_components: int = 2):
        self.svd = TruncatedSVD(n_components=n_components)

    def reduce(self, matrix: pd.DataFrame) -> pd.DataFrame:
        reduced_matrix = self.svd.fit_transform(matrix)
        return pd.DataFrame(
            reduced_matrix,
            index=matrix.index,
            columns=[f"component_{i+1}" for i in range(self.svd.n_components)]
        )

# Analyses

## Extract docs data

In [42]:
preprocessor = TextPreprocessor()
t_p = TextParser(category_mapping_path=os.path.join(DOCS_FOLDER_PATH, 'document_category_mapping.csv'))
tokens_df = t_p.parce()
tokens_df["tokens"] = tokens_df["raw_text"].apply(preprocessor.preprocess_text)
tokens_df

,file_name,raw_text,category,tokens
0,doc_4.txt,match team team coach team coach stadium playe...,sports,"[match, team, team, coach, team, coach, stadiu..."
1,doc_7.txt,network computer storage network network algor...,tech,"[network, computer, storage, network, network,..."
2,doc_3.txt,league player player team team training league...,sports,"[league, player, player, team, team, training,..."
3,doc_8.txt,match tournament league training football matc...,sports,"[match, tournament, league, training, football..."
4,doc_10.txt,code python performance storage storage algori...,tech,"[code, python, performance, storage, storage, ..."
5,doc_1.txt,election policy policy law minister reform ele...,politics,"[election, policy, policy, law, minister, refo..."
6,doc_2.txt,goal training player football football stadium...,sports,"[goal, training, player, football, football, s..."
7,doc_9.txt,performance data algorithm computer performanc...,tech,"[performance, datum, algorithm, computer, perf..."
8,doc_5.txt,match training stadium team coach goal trainin...,sports,"[match, training, stadium, team, coach, goal, ..."
9,doc_6.txt,football stadium training football match footb...,sports,"[football, stadium, training, football, match,..."


In [44]:
freq_analysis = FrequencyAnalysis(tokens_df)
word_doc_matrix = freq_analysis.build_word_document_matrix()
word_word_matrix = freq_analysis.build_word_word_matrix()
doc_category_matrix = freq_analysis.build_document_category_matrix()
doc_category_matrix

,politics,sports,tech
file_name,,,
doc_4.txt,False,True,False
doc_7.txt,False,False,True
doc_3.txt,False,True,False
doc_8.txt,False,True,False
doc_10.txt,False,False,True
doc_1.txt,True,False,False
doc_2.txt,False,True,False
doc_9.txt,False,False,True
doc_5.txt,False,True,False


## Metrics

In [43]:
euclidean_docs = freq_analysis.euclidean_distance(word_doc_matrix)
cosine_docs = freq_analysis.cosine_similarity_matrix(word_doc_matrix)
euclidean_docs

file_name,doc_4.txt,doc_7.txt,doc_3.txt,doc_8.txt,doc_10.txt,doc_1.txt,doc_2.txt,doc_9.txt,doc_5.txt,doc_6.txt
file_name,,,,,,,,,,
doc_4.txt,0.0,13.076697,6.855655,7.745967,14.832397,13.076697,4.242641,13.638182,5.09902,4.582576
doc_7.txt,13.076697,0.0,13.266499,13.0,5.91608,11.916375,11.874342,6.082763,11.269428,11.575837
doc_3.txt,6.855655,13.266499,0.0,9.848858,15.0,13.266499,6.855655,13.820275,7.28011,8.485281
doc_8.txt,7.745967,13.0,9.848858,0.0,14.764823,13.0,7.483315,13.56466,6.0,5.196152
doc_10.txt,14.832397,5.91608,15.0,14.764823,0.0,13.820275,13.784049,6.480741,13.266499,13.527749
doc_1.txt,13.076697,11.916375,13.266499,13.0,13.820275,0.0,11.874342,12.529964,11.269428,11.575837
doc_2.txt,4.242641,11.874342,6.855655,7.483315,13.784049,11.874342,0.0,12.489996,2.828427,5.0
doc_9.txt,13.638182,6.082763,13.820275,13.56466,6.480741,12.529964,12.489996,0.0,11.916375,12.206556
doc_5.txt,5.09902,11.269428,7.28011,6.0,13.266499,11.269428,2.828427,11.916375,0.0,4.358899


## Bayesian classifier

In [46]:
texts_for_classification = tokens_df["tokens"].apply(" ".join)
bayes_classifier = NaiveBayesClassifier()
bayes_classifier.train(texts_for_classification, tokens_df["category"])

### Text testing

In [48]:
test_texts = ["football match team coach", "algorithm data AI network"]
predictions = bayes_classifier.predict(test_texts)
predictions_proba = bayes_classifier.predict_proba(test_texts)
predictions_proba

[[0.0003214220105551502, 0.9996342802194875, 4.42977699569797e-05],
 [0.0019099740431377634, 0.00024157556499572545, 0.9978484503918664]]

In [60]:
svd_reducer = SVDReducer(n_components=2)
reduced_word_doc = svd_reducer.reduce(word_doc_matrix)
reduced_word_doc

,component_1,component_2
file_name,,
doc_4.txt,9.540787e+00,-5.782780e-16
doc_7.txt,9.445707e-16,7.739824e+00
doc_3.txt,8.389250e+00,1.800821e-16
doc_8.txt,8.265824e+00,-3.524815e-16
doc_10.txt,1.661230e-15,1.055349e+01
doc_1.txt,1.633624e-16,6.496615e-16
doc_2.txt,7.826117e+00,-1.098446e-16
doc_9.txt,1.151362e-15,8.469334e+00
doc_5.txt,7.063360e+00,-2.624031e-19


In [61]:
numeric_classifier = NumericNaiveBayesClassifier()
numeric_classifier.train(reduced_word_doc.values, tokens_df["category"])

test_reduced = reduced_word_doc.values[:3]  # приклад
predictions = numeric_classifier.predict(test_reduced)
probabilities = numeric_classifier.predict_proba(test_reduced)
probabilities

array([[0., 1., 0.],
       [0., 0., 1.],
       [0., 1., 0.]])